# ⚡ ViForge Quickstart: Fine-Tuning, Evaluating & Exporting Small Specialists

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vfcarida/ViForge/blob/main/notebooks/quickstart_colab.ipynb)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)

Welcome to **ViForge**! This interactive tutorial demonstrates how to:
1. **Audit Environment & Hardware**: Run `viforge doctor` to inspect GPU VRAM and accelerator status.
2. **Data Governance & Contamination Defense**: Screen datasets against benchmark leakage (HumanEval+, SWE-bench, GSM8K).
3. **Execute Post-Training Specialization**: Fine-tune an open-weights coding specialist (`Qwen/Qwen2.5-Coder-1.5B-Instruct`).
4. **Empirical Evaluation**: Measure domain gains and retention with 95% Wilson confidence intervals.
5. **Pareto Frontier Optimization**: Map candidates across Capability-per-Dollar, latency, and memory.
6. **Edge Deployment**: Export merged adapters to GGUF and generate ready-to-run Ollama Modelfiles.

## 1. Installation & Environment Diagnostics

First, clone ViForge and install dependencies.

In [ ]:
# Clone repository and install in editable mode
!git clone https://github.com/vfcarida/ViForge.git
%cd ViForge
!pip install -e ".[ui]" -q

In [ ]:
# Run environment diagnostics to inspect GPU, CUDA, and library versions
!viforge doctor

## 2. Contamination & Data Governance Shield

Detect whether training samples contain verbatim leakage of benchmark evaluation instances using the 10-gram sliding window shield.

In [ ]:
from viforge.preprocessing.contamination import ContaminationDetector

detector = ContaminationDetector(ngram_size=10, max_allowed_overlap=0.05)
detector.load_default_benchmark_banks()

# Check a candidate code snippet
clean_code = "def calculate_fibonacci(n: int) -> int:\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a"
overlaps = detector.check_sample(clean_code)
print("Benchmark overlap ratios:", overlaps)
assert all(ratio <= 0.05 for ratio in overlaps.values()), "Contamination detected!"

## 3. Experiment Manifest & Pre-Flight VRAM Profiling

Validate the experiment manifest schema and predict analytical VRAM consumption before allocating GPU memory.

In [ ]:
# Validate YAML manifest schema
!viforge validate configs/experiments/qwen2.5_coder_1.5b_quickstart.yaml

## 4. Run Specialization Experiment

Execute the end-to-end specialization pipeline:
- **Track A (`--mock`)**: Fast 10-second simulation tour using local deterministic fakes (ideal for CI/CD or free Colab CPU).
- **Track B (`--live`)**: Full training on real Hugging Face open weights (Qwen2.5-Coder-1.5B) using local GPU.

In [ ]:
# Run fast simulation tour
!viforge run configs/experiments/qwen2.5_coder_1.5b_quickstart.yaml --mock

## 5. Statistical Rigor (95% Wilson Confidence Intervals)

Calculate exact Wilson score intervals to verify whether evaluation gains are statistically significant.

In [ ]:
from viforge.metrics.statistical import WilsonScoreInterval

# Example: 45 passed out of 50 test problems
ci = WilsonScoreInterval.calculate(k_successes=45, n_trials=50, confidence_level=0.95)
print(f"Pass Rate: {ci['point_estimate']:.1%}")
print(f"95% Wilson Confidence Interval: [{ci['ci_lower']:.1%}, {ci['ci_upper']:.1%}]")

## 6. Multi-Objective Pareto Frontier Analysis

Compute non-dominated Pareto frontiers across Capability-per-Dollar, domain gain, latency, and VRAM.

In [ ]:
!viforge analyze configs/experiments/qwen2.5_coder_1.5b_quickstart.yaml

## 7. Export to GGUF & Ollama Modelfile

Convert trained LoRA checkpoints into merged standalone GGUF binaries for edge serving.

In [ ]:
# Export to GGUF format
!viforge export-gguf runs/deepseek_v4_pro_software_engineering_master/checkpoints/stage_1_sft \
  --output-dir exports/gguf \
  --quant-type Q4_K_M